In [1]:
import pandas as pd
from sklearn.preprocessing import MinMaxScaler, StandardScaler, PowerTransformer
from scipy.stats import skew
import numpy as np
import re
import os
from datetime import timedelta

In [2]:
datadir = '/scratch/hydro4/users/kv25483/MetricEvaluation/Data/'

### Create one dataframe containing all events for each resolution

In [3]:
all_events_5mins =[]
all_events_10mins =[]
all_events_30mins =[]
all_events_60mins =[]

total_events = 0
print(len(os.listdir(datadir + "DanishRainData_Outputs/5mins/")))

for num, file in enumerate(np.sort(os.listdir(datadir + "DanishRainData_Outputs/5mins/"))):
    if file != "All_events_571820_precip_minute.csv":
        # Get 5 mintue data
        df_5m = pd.read_csv(datadir + f"DanishRainData_Outputs/5mins/{file}")
        df_5m['event_num']=df_5m.index
        # Get rid of the trailing strings sometimes on the gauge_num
        df_5m['gauge_num'] = df_5m['gauge_num'].apply(
            lambda x: int(x.split('_')[0]) if isinstance(x, str) and x.split('_')[0].isdigit() else x)

        # Get other resolution data
        df_10m = pd.read_csv(datadir + f"DanishRainData_Outputs/{10}mins/{file}")
        df_30m = pd.read_csv(datadir + f"DanishRainData_Outputs/{30}mins/{file}")
        df_60m = pd.read_csv(datadir + f"DanishRainData_Outputs/{60}mins/{file}")

       # Find common event numbers in all four
        common_event_nums = (
            set(df_5m['event_num']) &
            set(df_10m['event_num']) &
            set(df_30m['event_num']) &
            set(df_60m['event_num']))

        # Filter all DataFrames to only include common events
        df_5m = df_5m[df_5m['event_num'].isin(common_event_nums)].reset_index(drop=True)
        df_10m = df_10m[df_10m['event_num'].isin(common_event_nums)].reset_index(drop=True)
        df_30m = df_30m[df_30m['event_num'].isin(common_event_nums)].reset_index(drop=True)
        df_60m = df_60m[df_60m['event_num'].isin(common_event_nums)].reset_index(drop=True)

        total_events = total_events + len(common_event_nums)
        print(f"{num} : {file}: {len(common_event_nums)} events retained across all resolutions, new total: {total_events}")    

        # Collect DataFrames into a dictionary for easy iteration
        dfs = {'5m': df_5m,
            '10m': df_10m,
            '30m': df_30m,
            '60m': df_60m}

        # Check the start and end times align
        for i in range(len(df_5m)):
            reference_start = pd.to_datetime(df_5m.iloc[i]['start_time'])
            reference_end = pd.to_datetime(df_5m.iloc[i]['end_time'])
            # print(reference_start,reference_end)
            for res, df in dfs.items():
                start = pd.to_datetime(df.iloc[i]['start_time'])
                end = pd.to_datetime(df.iloc[i]['end_time'])
                #print(res, start, end)

                delta_start = abs(start - reference_start)
                delta_end = abs(end - reference_end)

                if delta_start > timedelta(minutes=60):
                    print(f"⚠️ START mismatch in {res} at index {i}")
                    print(f"   {res} start: {start}, 5m start: {reference_start}, Δ: {delta_start}")

                if delta_end > timedelta(minutes=60):
                    print(f"⚠️ END mismatch in {res} at index {i}")
                    print(f"   {res} end: {end}, 5m end: {reference_end}, Δ: {delta_end}")

        all_events_5mins.append(df_5m)
        all_events_10mins.append(df_10m)             
        all_events_30mins.append(df_30m)
        all_events_60mins.append(df_60m)            

all_events_df_5mins = pd.concat(all_events_5mins)   
all_events_df_10mins = pd.concat(all_events_10mins)   
all_events_df_30mins = pd.concat(all_events_30mins)
all_events_df_60mins = pd.concat(all_events_60mins)   

for df in [all_events_df_5mins, all_events_df_10mins, all_events_df_30mins, all_events_df_60mins]:
    df.reset_index(drop=True, inplace=True)
    
print(len(all_events_df_5mins), len(all_events_df_10mins), len(all_events_df_30mins), len(all_events_df_60mins))

In [1]:
all_events_df_5mins

NameError: name 'all_events_df_5mins' is not defined

In [3]:
# def safe_parse_datetimes(df, file, resolution):
#     """Parse start/end times, returning None if parsing fails."""
#     try:
#         df['start_time'] = pd.to_datetime(df['start_time'], format='mixed')
#         df['end_time']   = pd.to_datetime(df['end_time'],   format='mixed')
#         return df
#     except Exception as e:
#         print(f"  ⚠️  Datetime parse failed for {resolution} in {file}: {e}")
#         return None

# # Track files that had to be skipped
# skipped_files = []

# files = np.sort(os.listdir(datadir + "DanishRainData_Outputs/5mins/"))[1:]
# skip  = {"All_events_571820_precip_minute.csv", 'All_events_5781_svk_precip_minute.csv'}
# print(f"Total files to process: {len(files)}")
# print(f"Skipping: {skip}\n")

# all_events_5mins  = []
# all_events_10mins = []
# all_events_30mins = []
# all_events_60mins = []
# total_events = 0


# for num, file in enumerate(files):
#     if file in skip:
#         print(f"[{num}] Skipping {file}")
#         continue

#     df_5m  = pd.read_csv(datadir + f"DanishRainData_Outputs/5mins/{file}")
#     df_10m = pd.read_csv(datadir + f"DanishRainData_Outputs/{10}mins/{file}")
#     df_30m = pd.read_csv(datadir + f"DanishRainData_Outputs/{30}mins/{file}")
#     df_60m = pd.read_csv(datadir + f"DanishRainData_Outputs/{60}mins/{file}")

#     print(f"[{num}] {file}")
#     print(f"  Loaded:  5m={len(df_5m)}  10m={len(df_10m)}  "
#           f"30m={len(df_30m)}  60m={len(df_60m)}")

#     # Clean gauge_num
#     df_5m['gauge_num'] = df_5m['gauge_num'].apply(
#         lambda x: int(x.split('_')[0]) if isinstance(x, str)
#         and x.split('_')[0].isdigit() else x)

#     # Parse datetimes safely — skip file if any resolution fails
#     parse_failed = False
#     for df, res in [(df_5m, '5m'), (df_10m, '10m'), (df_30m, '30m'), (df_60m, '60m')]:
#         result = safe_parse_datetimes(df, file, res)
#         if result is None:
#             parse_failed = True
#             break

#     if parse_failed:
#         print(f"  ⚠️  Skipping {file} due to datetime parse error")
#         skipped_files.append(file)
#         continue

#     # Floor start times to hour for matching
#     for df in [df_5m, df_10m, df_30m, df_60m]:
#         df['start_hour'] = df['start_time'].dt.floor('h')

#     common_starts = (
#         set(df_5m['start_hour']) &
#         set(df_10m['start_hour']) &
#         set(df_30m['start_hour']) &
#         set(df_60m['start_hour'])
#     )

#     print(f"  Before filter: 5m={len(df_5m)}  10m={len(df_10m)}  "
#           f"30m={len(df_30m)}  60m={len(df_60m)}")
#     print(f"  Common start times found: {len(common_starts)}")
#     print(f"  Dropped:  5m={len(df_5m)-len(common_starts)}  "
#           f"10m={len(df_10m)-len(common_starts)}  "
#           f"30m={len(df_30m)-len(common_starts)}  "
#           f"60m={len(df_60m)-len(common_starts)}")

#     df_5m  = df_5m[df_5m['start_hour'].isin(common_starts)].drop(
#         columns='start_hour').reset_index(drop=True)
#     df_10m = df_10m[df_10m['start_hour'].isin(common_starts)].drop(
#         columns='start_hour').reset_index(drop=True)
#     df_30m = df_30m[df_30m['start_hour'].isin(common_starts)].drop(
#         columns='start_hour').reset_index(drop=True)
#     df_60m = df_60m[df_60m['start_hour'].isin(common_starts)].drop(
#         columns='start_hour').reset_index(drop=True)

#     lengths = [len(df_5m), len(df_10m), len(df_30m), len(df_60m)]
#     if len(set(lengths)) != 1:
#         print(f"  ⚠️  LENGTH MISMATCH after filter: {lengths}")
#     else:
#         print(f"  ✓  All resolutions agree: {lengths[0]} events retained")

#     total_events += len(common_starts)
#     print(f"  Running total: {total_events}")

#     # Time alignment check
#     dfs = {'5m': df_5m, '10m': df_10m, '30m': df_30m, '60m': df_60m}
#     mismatches = 0
#     for i in range(len(df_5m)):
#         ref_start = df_5m.iloc[i]['start_time']
#         ref_end   = df_5m.iloc[i]['end_time']
#         for res, df in dfs.items():
#             if abs(df.iloc[i]['start_time'] - ref_start) > timedelta(minutes=60):
#                 print(f"  ⚠️  START mismatch [{res}] index={i}  "
#                       f"{res}:{df.iloc[i]['start_time']}  5m:{ref_start}")
#                 mismatches += 1
#             if abs(df.iloc[i]['end_time'] - ref_end) > timedelta(minutes=60):
#                 print(f"  ⚠️  END mismatch [{res}] index={i}  "
#                       f"{res}:{df.iloc[i]['end_time']}  5m:{ref_end}")
#                 mismatches += 1
#     if mismatches == 0:
#         print(f"  ✓  No time mismatches detected")
#     else:
#         print(f"  ⚠️  {mismatches} time mismatches detected")

#     print(f"  First event: 5m={df_5m.iloc[0]['start_time']}  "
#           f"10m={df_10m.iloc[0]['start_time']}  "
#           f"30m={df_30m.iloc[0]['start_time']}  "
#           f"60m={df_60m.iloc[0]['start_time']}")
#     print(f"  Last event:  5m={df_5m.iloc[-1]['start_time']}  "
#           f"10m={df_10m.iloc[-1]['start_time']}  "
#           f"30m={df_30m.iloc[-1]['start_time']}  "
#           f"60m={df_60m.iloc[-1]['start_time']}")

#     all_events_5mins.append(df_5m)
#     all_events_10mins.append(df_10m)
#     all_events_30mins.append(df_30m)
#     all_events_60mins.append(df_60m)

# # Concatenate
# all_events_df_5mins  = pd.concat(all_events_5mins).reset_index(drop=True)
# all_events_df_10mins = pd.concat(all_events_10mins).reset_index(drop=True)
# all_events_df_30mins = pd.concat(all_events_30mins).reset_index(drop=True)
# all_events_df_60mins = pd.concat(all_events_60mins).reset_index(drop=True)

# print("\n" + "="*60)
# print("FINAL SUMMARY")
# print("="*60)
# print(f"Total events retained: {total_events}")
# print(f"5m:  {len(all_events_df_5mins)}")
# print(f"10m: {len(all_events_df_10mins)}")
# print(f"30m: {len(all_events_df_30mins)}")
# print(f"60m: {len(all_events_df_60mins)}")
# all_equal = len(set([len(all_events_df_5mins), len(all_events_df_10mins),
#                      len(all_events_df_30mins), len(all_events_df_60mins)])) == 1
# print(f"All lengths equal: {'✓' if all_equal else '⚠️  MISMATCH'}")

# if skipped_files:
#     print(f"\nSkipped files ({len(skipped_files)}):")
#     for f in skipped_files:
#         print(f"  {f}")

Total files to process: 345
Skipping: {'All_events_571820_precip_minute.csv', 'All_events_5781_svk_precip_minute.csv'}

[0] All_events_500920_precip_minute.csv
  Loaded:  5m=940  10m=936  30m=931  60m=915
  Before filter: 5m=940  10m=936  30m=931  60m=915
  Common start times found: 912
  Dropped:  5m=28  10m=24  30m=19  60m=3
  ✓  All resolutions agree: 912 events retained
  Running total: 912
  ✓  No time mismatches detected
  First event: 5m=2012-04-06 00:15:00  10m=2012-04-06 00:10:00  30m=2012-04-06 00:00:00  60m=2012-04-06 00:00:00
  Last event:  5m=2024-12-30 02:05:00  10m=2024-12-30 02:00:00  30m=2024-12-30 02:00:00  60m=2024-12-30 02:00:00
[1] All_events_5012_svk_precip_minute.csv
  Loaded:  5m=291  10m=291  30m=281  60m=262
  Before filter: 5m=291  10m=291  30m=281  60m=262
  Common start times found: 262
  Dropped:  5m=29  10m=29  30m=19  60m=0
  ✓  All resolutions agree: 262 events retained
  Running total: 1174
  ✓  No time mismatches detected
  First event: 5m=1979-01-11 

[15] All_events_5057_svk_precip_minute.csv
  Loaded:  5m=2392  10m=2392  30m=2321  60m=2176
  Before filter: 5m=2392  10m=2392  30m=2321  60m=2176
  Common start times found: 2176
  Dropped:  5m=216  10m=216  30m=145  60m=0
  ✓  All resolutions agree: 2176 events retained
  Running total: 25676
  ✓  No time mismatches detected
  First event: 5m=1997-06-21 02:55:00  10m=1997-06-21 02:50:00  30m=1997-06-21 02:30:00  60m=1997-06-21 02:00:00
  Last event:  5m=2025-03-25 06:50:00  10m=2025-03-25 06:50:00  30m=2025-03-25 06:30:00  60m=2025-03-25 06:00:00
[16] All_events_5058_svk_precip_minute.csv
  Loaded:  5m=2345  10m=2344  30m=2273  60m=2130
  Before filter: 5m=2345  10m=2344  30m=2273  60m=2130
  Common start times found: 2130
  Dropped:  5m=215  10m=214  30m=143  60m=0
  ✓  All resolutions agree: 2130 events retained
  Running total: 27806
  ✓  No time mismatches detected
  First event: 5m=1997-09-06 12:55:00  10m=1997-09-06 12:50:00  30m=1997-09-06 12:30:00  60m=1997-09-06 12:00:00
  L

[30] All_events_5121_svk_precip_minute.csv
  Loaded:  5m=1780  10m=1780  30m=1729  60m=1620
  Before filter: 5m=1780  10m=1780  30m=1729  60m=1620
  Common start times found: 1620
  Dropped:  5m=160  10m=160  30m=109  60m=0
  ✓  All resolutions agree: 1620 events retained
  Running total: 45528
  ✓  No time mismatches detected
  First event: 5m=2005-09-14 07:45:00  10m=2005-09-14 07:40:00  30m=2005-09-14 07:30:00  60m=2005-09-14 07:00:00
  Last event:  5m=2025-03-25 07:35:00  10m=2025-03-25 07:30:00  30m=2025-03-25 07:30:00  60m=2025-03-25 07:00:00
[31] All_events_5122_svk_precip_minute.csv
  Loaded:  5m=1792  10m=1791  30m=1749  60m=1659
  Before filter: 5m=1792  10m=1791  30m=1749  60m=1659
  Common start times found: 1658
  Dropped:  5m=134  10m=133  30m=91  60m=1
  ✓  All resolutions agree: 1658 events retained
  Running total: 47186
  ✓  No time mismatches detected
  First event: 5m=2005-08-27 11:05:00  10m=2005-08-27 11:00:00  30m=2005-08-27 11:00:00  60m=2005-08-27 11:00:00
  La

[45] All_events_5163_svk_precip_minute.csv
  Loaded:  5m=298  10m=297  30m=285  60m=264
  Before filter: 5m=298  10m=297  30m=285  60m=264
  Common start times found: 264
  Dropped:  5m=34  10m=33  30m=21  60m=0
  ✓  All resolutions agree: 264 events retained
  Running total: 57605
  ✓  No time mismatches detected
  First event: 5m=2021-11-26 21:00:00  10m=2021-11-26 21:00:00  30m=2021-11-26 21:00:00  60m=2021-11-26 21:00:00
  Last event:  5m=2025-03-11 12:35:00  10m=2025-03-11 12:30:00  30m=2025-03-11 12:30:00  60m=2025-03-11 12:00:00
[46] All_events_5164_svk_precip_minute.csv
  Loaded:  5m=315  10m=314  30m=306  60m=287
  Before filter: 5m=315  10m=314  30m=306  60m=287
  Common start times found: 287
  Dropped:  5m=28  10m=27  30m=19  60m=0
  ✓  All resolutions agree: 287 events retained
  Running total: 57892
  ✓  No time mismatches detected
  First event: 5m=2021-11-26 20:45:00  10m=2021-11-26 20:40:00  30m=2021-11-26 20:30:00  60m=2021-11-26 20:00:00
  Last event:  5m=2025-03-25 

[61] All_events_5195_svk_precip_minute.csv
  Loaded:  5m=1398  10m=1397  30m=1367  60m=1287
  Before filter: 5m=1398  10m=1397  30m=1367  60m=1287
  Common start times found: 1287
  Dropped:  5m=111  10m=110  30m=80  60m=0
  ✓  All resolutions agree: 1287 events retained
  Running total: 78550
  ✓  No time mismatches detected
  First event: 5m=2009-09-28 04:15:00  10m=2009-09-28 04:10:00  30m=2009-09-28 04:00:00  60m=2009-09-28 04:00:00
  Last event:  5m=2025-02-24 08:15:00  10m=2025-02-24 08:10:00  30m=2025-02-24 08:00:00  60m=2025-02-24 08:00:00
[62] All_events_519920_precip_minute.csv
  Loaded:  5m=923  10m=922  30m=909  60m=889
  Before filter: 5m=923  10m=922  30m=909  60m=889
  Common start times found: 888
  Dropped:  5m=35  10m=34  30m=21  60m=1
  ✓  All resolutions agree: 888 events retained
  Running total: 79438
  ✓  No time mismatches detected
  First event: 5m=2013-11-01 06:30:00  10m=2013-11-01 06:30:00  30m=2013-11-01 06:30:00  60m=2013-11-01 06:00:00
  Last event:  5m=2

...

[276] All_events_5825_svk_precip_minute.csv
  Loaded:  5m=1357  10m=1357  30m=1300  60m=1193
  Before filter: 5m=1357  10m=1357  30m=1300  60m=1193
  Common start times found: 1193
  Dropped:  5m=164  10m=164  30m=107  60m=0
  ✓  All resolutions agree: 1193 events retained
  Running total: 336945
  ✓  No time mismatches detected
  First event: 5m=2008-04-16 18:50:00  10m=2008-04-16 18:50:00  30m=2008-04-16 18:30:00  60m=2008-04-16 18:00:00
  Last event:  5m=2025-03-25 12:40:00  10m=2025-03-25 12:40:00  30m=2025-03-25 12:30:00  60m=2025-03-25 12:00:00
[277] All_events_5830_svk_precip_minute.csv
  Loaded:  5m=1331  10m=1330  30m=1281  60m=1186
  Before filter: 5m=1331  10m=1330  30m=1281  60m=1186
  Common start times found: 1186
  Dropped:  5m=145  10m=144  30m=95  60m=0
  ✓  All resolutions agree: 1186 events retained
  Running total: 338131
  ✓  No time mismatches detected
  First event: 5m=2008-08-01 22:45:00  10m=2008-08-01 22:40:00  30m=2008-08-01 22:30:00  60m=2008-08-01 22:00:00


[291] All_events_588920_precip_minute.csv
  Loaded:  5m=913  10m=911  30m=897  60m=857
  Before filter: 5m=913  10m=911  30m=897  60m=857
  Common start times found: 854
  Dropped:  5m=59  10m=57  30m=43  60m=3
  ✓  All resolutions agree: 854 events retained
  Running total: 354073
  ✓  No time mismatches detected
  First event: 5m=2012-03-28 08:10:00  10m=2012-03-28 08:10:00  30m=2012-03-28 08:00:00  60m=2012-03-28 08:00:00
  Last event:  5m=2024-12-29 23:15:00  10m=2024-12-29 23:10:00  30m=2024-12-29 23:00:00  60m=2024-12-29 23:00:00
[292] All_events_5901_svk_precip_minute.csv
  Loaded:  5m=1552  10m=1551  30m=1495  60m=1408
  Before filter: 5m=1552  10m=1551  30m=1495  60m=1408
  Common start times found: 1408
  Dropped:  5m=144  10m=143  30m=87  60m=0
  ✓  All resolutions agree: 1408 events retained
  Running total: 355481
  ✓  No time mismatches detected
  First event: 5m=2006-08-13 08:20:00  10m=2006-08-13 08:20:00  30m=2006-08-13 08:00:00  60m=2006-08-13 08:00:00
  Last event:  

[306] All_events_5980_svk_precip_minute.csv
  Loaded:  5m=3361  10m=3361  30m=3236  60m=2988
  Before filter: 5m=3361  10m=3361  30m=3236  60m=2988
  Common start times found: 2988
  Dropped:  5m=373  10m=373  30m=248  60m=0
  ✓  All resolutions agree: 2988 events retained
  Running total: 374963
  ✓  No time mismatches detected
  First event: 5m=1979-01-07 19:20:00  10m=1979-01-07 19:20:00  30m=1979-01-07 19:00:00  60m=1979-01-07 19:00:00
  Last event:  5m=2021-06-04 01:15:00  10m=2021-06-04 01:10:00  30m=2021-06-04 01:00:00  60m=2021-06-04 01:00:00
[307] All_events_598120_precip_minute.csv
  Loaded:  5m=147  10m=147  30m=145  60m=140
  Before filter: 5m=147  10m=147  30m=145  60m=140
  Common start times found: 140
  Dropped:  5m=7  10m=7  30m=5  60m=0
  ✓  All resolutions agree: 140 events retained
  Running total: 375103
  ✓  No time mismatches detected
  First event: 5m=2023-03-12 17:40:00  10m=2023-03-12 17:40:00  30m=2023-03-12 17:30:00  60m=2023-03-12 17:00:00
  Last event:  5m

[322] All_events_607900_precip_minute.csv
  Loaded:  5m=475  10m=474  30m=470  60m=461
  Before filter: 5m=475  10m=474  30m=470  60m=461
  Common start times found: 460
  Dropped:  5m=15  10m=14  30m=10  60m=1
  ✓  All resolutions agree: 460 events retained
  Running total: 386213
  ✓  No time mismatches detected
  First event: 5m=2019-05-29 23:35:00  10m=2019-05-29 23:30:00  30m=2019-05-29 23:30:00  60m=2019-05-29 23:00:00
  Last event:  5m=2024-12-24 16:45:00  10m=2024-12-24 16:40:00  30m=2024-12-24 16:30:00  60m=2024-12-24 16:00:00
[323] All_events_608200_precip_minute.csv
  Loaded:  5m=101  10m=100  30m=99  60m=98
  Before filter: 5m=101  10m=100  30m=99  60m=98
  Common start times found: 98
  Dropped:  5m=3  10m=2  30m=1  60m=0
  ✓  All resolutions agree: 98 events retained
  Running total: 386311
  ✓  No time mismatches detected
  First event: 5m=2024-01-26 01:25:00  10m=2024-01-26 01:20:00  30m=2024-01-26 01:00:00  60m=2024-01-26 01:00:00
  Last event:  5m=2024-12-24 08:00:00 

[338] All_events_617000_precip_minute.csv
  Loaded:  5m=650  10m=649  30m=638  60m=614
  Before filter: 5m=650  10m=649  30m=638  60m=614
  Common start times found: 613
  Dropped:  5m=37  10m=36  30m=25  60m=1
  ✓  All resolutions agree: 613 events retained
  Running total: 390809
  ✓  No time mismatches detected
  First event: 5m=2017-08-15 20:00:00  10m=2017-08-15 20:00:00  30m=2017-08-15 20:00:00  60m=2017-08-15 20:00:00
  Last event:  5m=2024-12-30 10:40:00  10m=2024-12-30 10:40:00  30m=2024-12-30 10:30:00  60m=2024-12-30 10:00:00
[339] All_events_617400_precip_minute.csv
  Loaded:  5m=66  10m=65  30m=64  60m=60
  Before filter: 5m=66  10m=65  30m=64  60m=60
  Common start times found: 60
  Dropped:  5m=6  10m=5  30m=4  60m=0
  ✓  All resolutions agree: 60 events retained
  Running total: 390869
  ✓  No time mismatches detected
  First event: 5m=2024-03-05 16:50:00  10m=2024-03-05 16:50:00  30m=2024-03-05 16:30:00  60m=2024-03-05 16:00:00
  Last event:  5m=2024-12-23 01:40:00  10m

In [23]:
all_events_df_5mins.to_csv(datadir + 'IntermediateFiles/all_events_df_5mins_old.csv', index=False)
all_events_df_10mins.to_csv(datadir +  'IntermediateFiles/all_events_df_10mins_old.csv', index=False)
all_events_df_30mins.to_csv(datadir + 'IntermediateFiles/all_events_df_30mins_old.csv', index=False)
all_events_df_60mins.to_csv(datadir + 'IntermediateFiles/all_events_df_60mins_old.csv', index=False)

### Scale and transform

In [ ]:
all_events_df_5mins = pd.read_csv(datadir +  'IntermediateFiles/all_events_df_5mins_old.csv')
all_events_df_10mins = pd.read_csv(datadir +  'IntermediateFiles/all_events_df_10mins_old.csv')
all_events_df_30mins = pd.read_csv(datadir + 'IntermediateFiles/all_events_df_30mins_old.csv')
all_events_df_60mins = pd.read_csv(datadir + 'IntermediateFiles/all_events_df_60mins_old.csv')

/tmp/ipykernel_416301/3846318847.py:1: DtypeWarning: Columns (151) have mixed types. Specify dtype option on import or set low_memory=False.
  all_events_df_5mins = pd.read_csv(datadir +  'IntermediateFiles/all_events_df_5mins_old.csv')


In [6]:
# all_events_df_5mins = all_events_df_5mins.rename(columns={'lorentz_asymetry': 'lorenz_asymmetry'})
# all_events_df_10mins = all_events_df_10mins.rename(columns={'lorentz_asymetry': 'lorenz_asymmetry'})
# all_events_df_30mins = all_events_df_30mins.rename(columns={'lorentz_asymetry': 'lorenz_asymmetry'})
# all_events_df_60mins = all_events_df_60mins.rename(columns={'lorentz_asymetry': 'lorenz_asymmetry'})

# all_events_df_5mins = all_events_df_5mins.rename(columns={'lorentz_asymetry_DMC_10': 'lorenz_asymmetry_DMC_10'})
# all_events_df_10mins = all_events_df_10mins.rename(columns={'lorentz_asymetry_DMC_10': 'lorenz_asymmetry_DMC_10'})
# all_events_df_30mins = all_events_df_30mins.rename(columns={'lorentz_asymetry_DMC_10': 'lorenz_asymmetry_DMC_10'})
# all_events_df_60mins = all_events_df_60mins.rename(columns={'lorentz_asymetry_DMC_10': 'lorenz_asymmetry_DMC_10'})

### Remove rows with unrealistically high precip

In [ ]:
bad_indices = all_events_df_5mins.sort_values(by="total_precip", ascending=False)[:15].index

# Drop rows at these indices from all dataframes
all_events_df_5mins = all_events_df_5mins.drop(index=bad_indices)
all_events_df_10mins = all_events_df_10mins.drop(index=bad_indices)
all_events_df_30mins = all_events_df_30mins.drop(index=bad_indices)
all_events_df_60mins = all_events_df_60mins.drop(index=bad_indices)

for df in [all_events_df_5mins, all_events_df_10mins, all_events_df_30mins, all_events_df_60mins]:
    df.reset_index(drop=True, inplace=True)
    
print(len(all_events_df_5mins), len(all_events_df_10mins), len(all_events_df_30mins), len(all_events_df_60mins))

### Remove very small events

In [25]:
print(len(all_events_df_5mins))
all_events_df_5mins = all_events_df_5mins[all_events_df_5mins['total_precip']>4].copy()
print(len(all_events_df_5mins))

print(len(all_events_df_10mins))
all_events_df_10mins = all_events_df_10mins[all_events_df_10mins['total_precip']>4].copy()
print(len(all_events_df_10mins))

print(len(all_events_df_30mins))
all_events_df_30mins = all_events_df_30mins[all_events_df_30mins['total_precip']>4].copy()
print(len(all_events_df_30mins))

print(len(all_events_df_60mins))
all_events_df_60mins = all_events_df_60mins[all_events_df_60mins['total_precip']>4].copy()
print(len(all_events_df_60mins))

397157
240579
397157
240590
397157
240609
397157
240588


# Create version for all resolutions, just raw
### Specify just raw columns to keep

In [26]:
raw_cols = []
for col in all_events_df_5mins.columns:
    if not col.endswith('_dmc') and not col.endswith('dblnorm'):
        raw_cols.append(col)

### Also remove 'frac_q1', 'frac_q2', 'frac_q3', 'frac_q4'
Because these bring lots of nans (with current calculation method)  
ni is the same as peak mean ratio

In [31]:
cols_to_del = ['frac_q1', 'frac_q2', 'frac_q3', 'frac_q4', 'BSC', 'ni', 'm1', 'm2', 'm3', 'm4', 'm5']
more_cols_to_del = ['gauge_num',  'start_time', 'end_time', 'min_intensity', 'heaviest_half',
                    'T50', 'BSC_Index', 'peak_mean_ratio_scaled',"relative_amp_scaled"]
                   
raw_cols = [x for x in raw_cols if x not in cols_to_del]  
raw_cols = [x for x in raw_cols if x not in more_cols_to_del]    

In [28]:
all_events_df_5mins_raw = all_events_df_5mins[raw_cols].copy()
all_events_df_10mins_raw = all_events_df_10mins[raw_cols].copy()
all_events_df_30mins_raw = all_events_df_30mins[raw_cols].copy()
all_events_df_60mins_raw = all_events_df_60mins[raw_cols].copy()

In [29]:
all_events_df_60mins_raw["I30"] = 0

In [37]:
print(len(all_events_df_5mins_raw), len(all_events_df_10mins_raw), len(all_events_df_30mins_raw), len(all_events_df_60mins_raw))

240579 240590 240609 240588


In [32]:
print(len(all_events_df_5mins_raw), len(all_events_df_10mins_raw), len(all_events_df_30mins_raw), len(all_events_df_60mins_raw))

bad_indices_all = []
for all_events_df in [all_events_df_5mins_raw, all_events_df_10mins_raw, all_events_df_30mins_raw, all_events_df_60mins_raw]:
    columns_with_nan = all_events_df.columns[all_events_df.isnull().any()].tolist()
    print("Columns with at least one NaN value:")
    print(columns_with_nan)
    # #Find indices of rows with any NaNs in the 60-minute data
    bad_indices = all_events_df[all_events_df.isnull().any(axis=1)].index
    bad_indices_lst = bad_indices.tolist()
    print(len(bad_indices_lst))
    if len(bad_indices) >0:
        bad_indices_all.extend(bad_indices_lst)
    
# Drop rows at these indices from all dataframes
all_events_df_5mins_raw = all_events_df_5mins_raw.drop(index=bad_indices_all)
all_events_df_10mins_raw = all_events_df_10mins_raw.drop(index=bad_indices_all)
all_events_df_30mins_raw = all_events_df_30mins_raw.drop(index=bad_indices_all)
all_events_df_60mins_raw = all_events_df_60mins_raw.drop(index=bad_indices_all)   

print(len(all_events_df_5mins_raw), len(all_events_df_10mins_raw), len(all_events_df_30mins_raw), len(all_events_df_60mins_raw))

240579 240590 240609 240588
Columns with at least one NaN value:
['heaviest_half', 'heaviest_half_DMC_10']
237765
Columns with at least one NaN value:
['heaviest_half', 'heaviest_half_DMC_10']
237929
Columns with at least one NaN value:
['heaviest_half', 'heaviest_half_DMC_10']
238524
Columns with at least one NaN value:
['kurtosis', 'asymm_d', 'heaviest_half', 'heaviest_half_DMC_10']
238830


KeyError: '[649, 1117, 1836, 8099, 9158, 9888, 9938, 31932, 32404, 35142, 36310, 54447, 54605, 54636, 59139, 72706, 73861, 82420, 83715, 87022, 88769, 89692, 118083, 118594, 132783, 136388, 143137, 147709, 151854, 162857, 164348, 182607, 184603, 188121, 188700, 188965, 194757, 201943, 204420, 208207, 220459, 223487, 258271, 296554, 326964, 347511, 357441, 360383, 374505, 374821, 375361, 379693, 382572, 384716, 385531, 387658, 387904, 388442, 390490, 392219, 392289, 392914, 393839, 393905, 396021, 396096, 483, 688, 1863, 1925, 8099, 9729, 9893, 10215, 33645, 36013, 36017, 38127, 38938, 40527, 40538, 52183, 54267, 54605, 55168, 57415, 57437, 59139, 80140, 82420, 82709, 83291, 83476, 83899, 88262, 88769, 89692, 118060, 119577, 131686, 133163, 136220, 141850, 142019, 142493, 143207, 143281, 143459, 147709, 158691, 158761, 162825, 175106, 185041, 188609, 194757, 202209, 204631, 211764, 212515, 222362, 222896, 223234, 260699, 267023, 269103, 294292, 296554, 357796, 374821, 375154, 379052, 37

### Join all into one dataframe

In [ ]:
metric_columns =all_events_df_5mins_raw.columns

In [ ]:
all_events_df_5mins_raw = all_events_df_5mins_raw.add_suffix('_5m')
all_events_df_10mins_raw = all_events_df_10mins_raw.add_suffix('_10m')
all_events_df_30mins_raw = all_events_df_30mins_raw.add_suffix('_30m')
all_events_df_60mins_raw = all_events_df_60mins_raw.add_suffix('_60m')

In [ ]:
all_events_all_res = pd.concat([all_events_df_5mins_raw, all_events_df_10mins_raw,
                               all_events_df_30mins_raw, all_events_df_60mins_raw], axis=1)
# Step 2: Define the event number columns from each resolution
event_cols = ['event_num_5m', 'event_num_10m', 'event_num_30m', 'event_num_60m']

# Step 3: Drop rows with any missing event numbers
all_events_all_res = all_events_all_res.dropna(subset=event_cols)

# # Step 4: Keep only rows where all event numbers are equal (same event matched across resolutions)
# all_events_all_res = all_events_all_res[all_events_all_res[event_cols].nunique(axis=1) == 1]

In [ ]:
all_events_all_res[['event_num_5m',  'event_num_30m', 'event_num_60m']]

In [ ]:
all_events_all_res.to_csv(datadir + "IntermediateFiles/NotScaled_AllRes.csv",index=False)

# Create version for 5 minutes, DMC and raw
### Specify columns to keep

In [ ]:
cols_to_del = ['frac_q1', 'frac_q2', 'frac_q3', 'frac_q4', 'BSC', 'ni', 'T50', 'm1', 'm2', 'm3', 'm4', 'm5', 'BSC_Index']
more_cols_to_del = ['gauge_num', 'start_time', 'end_time', 'event_num', 'heaviest_half_raw']

all_to_del = cols_to_del + more_cols_to_del
all_columns = df.columns.tolist()
cols_to_keep = [col for col in all_columns if col not in all_to_del]

# Remove any column that matches exactly or starts with one of the names (e.g., has suffixes like _DMC_10)
cols_to_keep = [col for col in cols_to_keep if col not in all_to_del]

# Then filter the DataFrame
all_events_df_5mins_filtered = all_events_df_5mins[cols_to_keep].copy()

In [ ]:
all_events_df_5mins_filtered['mean_intensity_dmc'] = all_events_df_5mins_filtered['mean_intensity_dmc']/10

In [ ]:
all_events_df_5mins_filtered.to_csv(datadir + "IntermediateFiles/NotScaled_RawVsDMC.csv",index=False)